In [ ]:
# Data Exploration - Airbnb Tokyo
import sys
sys.path.append('..')

from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.data.feature_engineer import FeatureEngineer

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# 1. Load Data
loader = DataLoader()
raw_data = loader.load_all(sample_listings=None)  # Load all listings

# 2. Clean Data
cleaner = DataCleaner()
cleaned_data = cleaner.clean_all(raw_data)

# 3. Engineer Features
engineer = FeatureEngineer()
final_data = engineer.engineer_all(cleaned_data)

# 4. Explore Listings
df_listings = final_data['listings']
print(f"Listings: {len(df_listings):,} rows, {len(df_listings.columns)} columns")
df_listings.head()

# 5. Basic Statistics
df_listings.describe()

# 6. Price Distribution
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
df_listings['price'].hist(bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Price ($)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
df_listings['price'].hist(bins=50, edgecolor='black', range=(0, 500))
plt.title('Price Distribution (0-500)')
plt.xlabel('Price ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# 7. Price by Room Type
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_listings, x='room_type', y='price')
plt.title('Price by Room Type')
plt.ylim(0, 500)
plt.show()

# 8. Map
# Create a folium map
map_center = [df_listings['latitude'].mean(), df_listings['longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=11)

# Add sample of listings (every 50th to avoid clutter)
sample = df_listings.iloc[::50]
for _, row in sample.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        popup=f"${row['price']:.0f} - {row['room_type']}",
        color='blue' if row['price'] < 100 else 'red',
        fill=True
    ).add_to(m)

m.save('map.html')
print("Map saved as map.html")

# 9. Summary
print(f"\n Dataset Summary:")
print(f"Total listings: {len(df_listings):,}")
print(f"Average price: ${df_listings['price'].mean():.2f}")
print(f"Median price: ${df_listings['price'].median():.2f}")
print(f"Price range: ${df_listings['price'].min():.2f} - ${df_listings['price'].max():.2f}")